In [1]:
# !pip install -q chromadb
# !pip install -q sentence-transformers
# !pip install -q gradio
# !pip install --upgrade numpy ml_dtypes transformers sentence-transformers
# !pip install -q rank-bm25

In [1]:
import os
import re

import gradio as gr
import chromadb
import pickle
import numpy as np

from datetime import datetime
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from google import genai
from google.genai import types

from sentence_transformers import SentenceTransformer

from rank_bm25 import BM25Okapi

import json

ModuleNotFoundError: No module named 'rank_bm25'

In [3]:
load_dotenv(dotenv_path="local.env")

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Security Error: GEMINI_API_KEY environment variable is missing!")

client = genai.Client()

In [4]:
OUTPUT_DIR = "mining_act_html_versions"
model = "gemini-3.1-flash-lite"

list_of_target_language = ['English',
                            'Spanish',
                              'Hindi',
                              'Tamil',
                              'French',
                              'German',
                               'Italian',
                               'Polish'
                              ]

In [5]:
print("Loading local embedding model ('all-MiniLM-L6-v2')...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading local embedding model ('all-MiniLM-L6-v2')...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
DB_STORAGE_PATH = "./chroma_db_storage"
os.makedirs(DB_STORAGE_PATH, exist_ok=True)

# 2. Bind Persistent Local Vector Client
chroma_client = chromadb.PersistentClient(path=DB_STORAGE_PATH)
collection_name = "latest_mining_act_rag"

# Fetch or spin up the collection database
collection = chroma_client.get_or_create_collection(name=collection_name)

print(f"Local Persistent Client connected at: '{DB_STORAGE_PATH}'")
print(f"Active collection profile context: '{collection_name}' (Current Size: {collection.count()} blocks)")
print("-" * 70)

Local Persistent Client connected at: './chroma_db_storage'
Active collection profile context: 'latest_mining_act_rag' (Current Size: 0 blocks)
----------------------------------------------------------------------


In [7]:
def find_latest_version(directory):
    if not os.path.exists(directory):
        raise FileNotFoundError(f"Directory '{directory}' is missing.")
    all_versions = []
    for filename in os.listdir(directory):
        if filename.startswith("Mining Act 1978") and filename.endswith(".html"):
            file_path = os.path.join(directory, filename)
            match = re.search(r"Mining Act 1978_.+?_(.+?)\.html", filename)
            if match:
                date_str = match.group(1).replace("_", " ")
                parsed_date = datetime.strptime(date_str, "%d %b %Y")
                all_versions.append({"path": file_path, "date": parsed_date})
    all_versions.sort(key=lambda x: x["date"], reverse=True)
    return all_versions[0]["path"]

def extract_clean_text(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    content_area = soup.find("section", id="main") or soup.find("section", id="outer")
    target_soup = content_area if content_area else soup
    for element in target_soup(["script", "style", "meta", "link", "header", "footer", "nav", "form"]):
        element.decompose()
    text = target_soup.get_text(separator="\n")
    return "\n".join([line.strip() for line in text.splitlines() if line.strip()])

def split_into_chunks(text, chunk_size=10000, chunk_overlap = 10000 * 0.2):
    # Smaller chunks (10k chars) are better optimized for local embedding semantic resolution
    if chunk_overlap >= chunk_size:
        raise ValueError("Chunk overlap must be smaller than the chunk size.")
        
    lines = text.splitlines()
    chunks = []
    
    current_chunk = []
    current_size = 0
    
    i = 0
    while i < len(lines):
        line = lines[i]
        current_chunk.append(line)
        current_size += len(line) + (1 if current_chunk else 0)
        
        if current_size >= chunk_size:
            chunks.append("\n".join(current_chunk))
            
            overlap_size = 0
            overlap_lines = []
            
            for j in range(i, -1, -1):
                line_len = len(lines[j]) + 1
                if overlap_size + line_len <= chunk_overlap:
                    overlap_size += line_len
                    overlap_lines.insert(0, lines[j])
                else:
                    break
            
            current_chunk = overlap_lines
            current_size = overlap_size
            
        i += 1
        
    if current_chunk:
        chunks.append("\n".join(current_chunk))
        
    return chunks

In [8]:
latest_file = find_latest_version(OUTPUT_DIR)
pure_text = extract_clean_text(latest_file)
document_chunks = split_into_chunks(pure_text)

DB_STORAGE_PATH = "./chroma_local_embed_storage"
chroma_client = chromadb.PersistentClient(path=DB_STORAGE_PATH)
collection = chroma_client.get_or_create_collection(name="mining_act_local_embeddings")

In [9]:
with open("RAG_current_mining_act_file.pkl", "rb") as file:
    RAG_current_mining_act_file = pickle.load(file)
print(f"Current loaded mining act file: {RAG_current_mining_act_file}")

if latest_file == RAG_current_mining_act_file:
    print(f"-> Same Local disk vector cache found. Skipping embedding generation.")
else:
    print(f"Embedding {len(document_chunks)} chunks completely offline using local CPU/GPU...")
    for idx, chunk in enumerate(document_chunks):
        vector = embedding_model.encode(chunk).tolist()
        
        collection.add(
            embeddings=[vector],
            documents=[chunk],
            ids=[f"block_{idx}"]
        )
    print("-> Local database initialization complete and committed to disk!")
    with open("RAG_current_mining_act_file.pkl", "wb") as file:
        pickle.dump(latest_file, file)

Current loaded mining act file: mining_act_html_versions/Mining Act 1978_09-q0-00_28_May_2026.html
-> Same Local disk vector cache found. Skipping embedding generation.


In [10]:
def ask_hybrid_rag(user_query: str):
    # Vectorize the question locally using the same local sentence-transformer model
    query_vector = embedding_model.encode(user_query).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=20)
    retrieved_context = "\n\n---\n\n".join(results['documents'][0])
    
    rag_prompt = f"""
                    You are an expert legal assistant. 
                    Answer the user question using ONLY the provided verified context below.
                    If the context does not contain the answer, 
                    respond exactly with: "I don't know based on the provided context."
                    
                    ---
                    [VERIFIED CONTEXT]
                    {retrieved_context}
                    ---
                    
                    [QUESTION]
                    {user_query}
                """

    # print(rag_prompt)
    response = client.models\
                    .generate_content(
                                    model=model,
                                    contents=rag_prompt,
                                    config=types.GenerateContentConfig(
                                        temperature=0.0  
                                    )
                                )
    
    return response.text

In [11]:
def get_hybrid_context(user_query: str, collection, n_results=10):
    """
    Combines Vector Search (ChromaDB) and Keyword Search (BM25) 
    using Reciprocal Rank Fusion (RRF).
    """
    # 1. Fetch all documents currently stored in ChromaDB to build the BM25 index
    db_data = collection.get()
    all_documents = db_data["documents"]
    
    if not all_documents:
        return ""

    # 2. KEYWORD SEARCH (BM25)
    # Tokenize the corpus (lowercase split is a simple, effective tokenizer for this)
    tokenized_corpus = [doc.lower().split() for doc in all_documents]
    bm25 = BM25Okapi(tokenized_corpus)
    
    tokenized_query = user_query.lower().split()
    # Get BM25 scores for all documents
    bm25_scores = bm25.get_scores(tokenized_query)
    # Sort document indices by highest BM25 score
    top_keyword_indices = np.argsort(bm25_scores)[::-1][:n_results]
    
    # 3. VECTOR SEARCH (ChromaDB)
    query_vector = embedding_model.encode(user_query).tolist()
    vector_res = collection.query(query_embeddings=[query_vector], n_results=n_results)
    vector_documents = vector_res['documents'][0] if vector_res['documents'] else []

    # 4. RECIPROCAL RANK FUSION (RRF)
    # Assigns a combined score based on how high a document ranks in both systems
    rrf_scores = {doc: 0.0 for doc in all_documents}
    k = 60 # Standard constant ranking constant for RRF
    
    # Rank vector results
    for rank, doc in enumerate(vector_documents):
        if doc in rrf_scores:
            rrf_scores[doc] += 1.0 / (k + rank + 1)
            
    # Rank keyword results
    for rank, idx in enumerate(top_keyword_indices):
        doc = all_documents[idx]
        rrf_scores[doc] += 1.0 / (k + rank + 1)
        
    # Sort all documents by their combined RRF score
    hybrid_sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    # Take the top unique results up to your limit
    top_hybrid_docs = [doc for doc, score in hybrid_sorted_docs[:n_results]]
    
    return "\n\n---\n\n".join(top_hybrid_docs)

def ask_hybrid_rag_with_memory(user_message: str, 
                               chat_history: list,
                              language: str):
    """
    Processes chat conversations by vectorizing the latest question,
    assembling past chat memory strings, and prompting Gemini.
    """
    if not user_message.strip():
        return "Please enter a valid legal question."
        
    query_vector = embedding_model.encode(user_message).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=10)
    retrieved_context = get_hybrid_context(user_message, collection, n_results=20)
    
    formatted_history = ""
    for turn in chat_history:
        role = "User" if turn["role"] == "user" else "Assistant"
        formatted_history += f"{role}: {turn['content']}\n"
        
    rag_prompt = f"""
                    You are an expert legal assistant specializing in the Western Australian Mining Act 1978. 
                    Your strict core rule is to answer the user question using ONLY the provided verified legal context below. 
                    
                    If the context does not contain the answer, or if a follow-up question refers to information 
                    outside of the provided context, respond exactly with: "I don't know based on the provided context." 

                    it is very important that you Do not fabricate the CONTEXT.
                    it is very important that you Keep outputs realistic and aligned with industry standards
                    it is very important that you Ensure all suggestions are practical and implementable
                    It is very important to use clean MS word text without any markdown formula 

                    ---
                    [VERIFIED LEGAL CONTEXT]
                    {retrieved_context}
                    ---
                    
                    [CONVERSATIONAL HISTORY]
                    {formatted_history}
                    ---
                    
                    [NEW USER QUESTION]
                    {user_message}
                    """

    response = client.models.generate_content(
        model=model,
        contents=rag_prompt, 
        config=types.GenerateContentConfig(
            temperature=0.0  # Keep at 0.0 to strictly prevent hallucinations
        )
    )

    master_response = response.text

    if language != "English":
        response_tranlate_prompt = f"""
                                You are an expert legal translator and legislative archivist fluent in both 
                                English and formal {language} legal terminology.
                                
                                ---
                                ### Strict Content & Formatting Guidelines:
                                - Do not interpret, expand, or summarize the text. 
                                Translate the technical legal content exactly as written.
                                - Maintain the exact markdown layout, including headings (#, ##, ###), bullet lists, 
                                and markdown table structures.
                                - Keep proper nouns, citations, alphanumeric codes, 
                                and official statutory references (e.g., "Mining Act 1978", "Part 4AA", "Section 103AV") 
                                in their original format unless a direct, universally accepted legal equivalent exists in {language}.
                                - Ensure the final output is completely clean and formatted entirely in Markdown. 
                                Do not include any conversational introductions, meta-commentary, or closing remarks.
                                ---
                                
                                ### REPORT TO TRANSLATE to {language} :
                                {master_response}
                                """
        
        response = client.models.generate_content(
        model=model,
        contents=response_tranlate_prompt, 
        config=types.GenerateContentConfig(
            temperature=0.0  # Keep at 0.0 to strictly prevent hallucinations
            )
        )
        
        master_response = response.text
        
    return master_response

In [14]:
with gr.Blocks() as demo:
    # Title and Description placed at the very top
    gr.Markdown("# ⚖️ Mining Act 1978 Legal AI")
    gr.Markdown("Ask complex legislative questions. This assistant maintains conversational context and checks against local vector indices.")
    
    # Place your dropdown right here (below description, above chat)
    language_dropdown = gr.Dropdown(
        choices=list_of_target_language,
        value="English",
        label="Response Language",
        info="Select the language you want the legal assistant to reply in."
    )
    
    # 2. Instantiate the ChatInterface inside the block
    gr.ChatInterface(
        fn=ask_hybrid_rag_with_memory, 
        # Pass the pre-defined dropdown component into additional_inputs
        additional_inputs=[language_dropdown],
        examples=[
            ["What are the rules regarding the land subject to an exploration licence?", "English"],
            ["Can you elaborate on the surrender requirements mentioned in your last answer?", "English"],
            ["What is Part 4AA about?", "English"],
            ["What the content list for Mining Act 1978", "English"]
        ]
    )

demo.launch(inline=True, share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
